# Proyecto Isla Verde — Punto de Entrada Reproducible

Ejecuta y verifica el pipeline local completo: **datos ICE → grafo → QUBO/Ising → baselines → QAOA/Qulacs → comparación**.

El perfil `smoke` comprueba todo rápidamente con MVP-8, p=1 y una corrida. Cambie a `report` para p=1,2,3, cinco corridas y las tres escalas. Ningún perfil envía trabajos a H2.

## I. Configuración

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "main.py").exists() and (ROOT.parent / "main.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("XDG_CONFIG_HOME", str(ROOT / "scratch" / ".config"))

PROFILE = "smoke"       # "smoke" o "report"
INSTALL_DEPS = False     # True solo en un entorno nuevo
SEED = 42
print("Raíz:", ROOT, "| perfil:", PROFILE, "| semilla:", SEED)

In [ ]:
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
else:
    print("Instalación omitida; usando el entorno activo.")

## II. Pipeline completo

`main.py` orquesta modelado, autoverificación QUBO/Ising, baselines independientes, QAOA local y comparación sobre la misma instancia.

In [ ]:
subprocess.run([
    sys.executable, "main.py", "--profile", PROFILE,
    "--data-dir", "data", "--out-dir", "scratch",
    "--seed", str(SEED), "--skip-install",
], check=True)

## III. Auditoría automática

El resumen solo declara `PASS` después de comprobar artefactos, óptimo exacto, baselines, identidad de instancia clásica/cuántica, razones en [0,1] y 2p parámetros por circuito. `smoke` es diagnóstico; solo `report` satisface el mínimo estadístico de cinco corridas.

In [ ]:
scratch = ROOT / "scratch"
summary = json.loads((scratch / "pipeline_summary.json").read_text(encoding="utf-8"))
assert summary["status"] == "PASS"
assert summary["external_jobs_submitted"] is False
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
import pandas as pd

rows = []
index = json.loads((scratch / "isla_verde_index.json").read_text(encoding="utf-8"))
qaoa = json.loads((scratch / "qaoa_barrido_p.json").read_text(encoding="utf-8"))["results"]
for tier, filename in index["instances"].items():
    data = json.loads((scratch / filename).read_text(encoding="utf-8"))
    baseline = data["baselines"]["maxcut"]
    optimum = baseline["brute_force"]["cut"]
    for result in qaoa.get(tier, []):
        rows.append({
            "instancia": tier, "qubits": len(data["variable_order"]), "p": result["p"],
            "greedy r": baseline["greedy"]["cut"] / optimum,
            "GW r": baseline["goemans_williamson"]["cut"] / optimum,
            "QAOA r media": result["ratio_mean"], "QAOA r std": result["ratio_std"],
        })
pd.DataFrame(rows)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(scratch / "qaoa_ratio_vs_p.png"), width=650))
display(Image(filename=str(scratch / "comparacion_qaoa_vs_clasico.png"), width=650))